# LangChain Memory 시리즈 3/7: ConversationTokenBufferMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 대화 횟수(k)가 아닌 **토큰 수** 기준으로 메모리를 제한해야 하는 이유를 설명할 수 있어요
- `max_token_limit` 파라미터로 메모리 상한선을 설정할 수 있어요
- `ConversationBufferWindowMemory`와 비교해 어느 상황에 무엇을 써야 할지 판단할 수 있어요
- LLM 인스턴스가 토큰 계산에 왜 필요한지 이해할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| ConversationBufferWindowMemory | 최근 k개 대화 턴을 유지하는 메모리 (2번 노트북) |
| 토큰 (Token) | LLM이 텍스트를 처리하는 최소 단위 |
| 토크나이저 (Tokenizer) | 텍스트를 토큰으로 분리하는 도구 |

## k 기반 vs 토큰 기반, 차이가 뭘까요?

`ConversationBufferWindowMemory`는 **대화 턴 수**로 메모리를 제한해요. 그런데 대화 1턴이 10토큰일 수도, 500토큰일 수도 있어요. 메시지 길이가 일정하지 않다면 k 기반 제한은 정확하지 않아요.

`ConversationTokenBufferMemory`는 **실제 토큰 수**를 측정해서 제한해요. LLM의 토크나이저를 사용하기 때문에 API 요금 계산에 쓰이는 토큰 수와 정확히 일치해요.

```mermaid
flowchart TD
    subgraph Compare["k 기반 vs 토큰 기반 비교"]
        MSG1["짧은 메시지 × 3턴<br/>(총 30토큰)"]
        MSG2["긴 메시지 × 3턴<br/>(총 900토큰)"]
    end

    MSG1 & MSG2 -->|k=3이면 동일하게 유지| K_MEM[WindowMemory]
    MSG1 -->|30토큰 < 100 제한, 유지| T_MEM[TokenBufferMemory]
    MSG2 -->|900토큰 > 100 제한, 삭제| T_MEM

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    class MSG1 input
    class MSG2 error
    class K_MEM,T_MEM process
```

> **참고**: 이 노트북은 레거시 메모리 클래스를 사용해요. LangChain 1.0.x에서는 토큰 트리밍 미들웨어 패턴이 권장돼요. 최신 방식은 13번 노트북을 참고하세요.

In [1]:
# ---------------------------------------------------
# 환경 변수 로드 및 레거시 메모리 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
from langchain.memory import ConversationTokenBufferMemory
# ChatOpenAI: 토큰 계산에 쓰이는 LLM의 토크나이저를 제공
from langchain_openai import ChatOpenAI

load_dotenv()

True

## 1. 기본 사용법 - 토큰 제한 설정

`max_token_limit` 파라미터로 저장할 최대 토큰 수를 지정합니다.  
토큰 수가 이 제한을 초과하면 오래된 메시지부터 자동으로 삭제됩니다.


### 💡 토큰 기반 메모리 초기화 흐름

```mermaid
graph TD
    ENV["환경 변수 로드 및 llm 생성"] --> MEM_INIT[ConversationTokenBufferMemory 생성]
    MEM_INIT --> PARAMS["max_token_limit=150,
return_messages=True"]
    PARAMS --> READY[토큰 기반 history 준비]
```

In [2]:
# ============================================================
# TODO: ConversationTokenBufferMemory를 max_token_limit=150으로 생성하세요
# 힌트: ConversationTokenBufferMemory(llm=llm, max_token_limit=150, return_messages=True)
# 예상 결과: 대화가 쌓여 150 토큰을 초과하면 오래된 메시지가 자동 삭제됨
# ============================================================

# 1단계: LLM 모델 생성 (토큰 계산에 사용)
# - llm 파라미터: 실제 토크나이저를 제공하는 필수 인자
# - ConversationTokenBufferMemory는 이 llm의 토크나이저로 토큰 수를 측정함
llm = ChatOpenAI(model="gpt-4o-mini")

# 2단계: 최대 토큰 길이를 150으로 제한하는 메모리 생성
# - max_token_limit=150: history 토큰 합계가 150을 넘으면 오래된 메시지부터 제거
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=150,
    return_messages=True
)

/var/folders/k2/q6_f_rp50lqg7l24d_3dk1vh0000gn/T/ipykernel_99054/1658790782.py:14: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationTokenBufferMemory(


### 💡 토큰 제한 기반 저장/트리밍

```mermaid
flowchart TD
    subgraph SaveLoop[여러 턴 저장]
        H[사용자 발화] --> SAVE["save_context()"]
        A[AI 응답] --> SAVE
        SAVE --> HIST[history 누적]
    end

    HIST --> COUNT[LLM 토크나이저로 토큰 길이 계산]
    COUNT --> CHECK{총 토큰 > 150?}
    CHECK -- 예 --> DROP[가장 오래된 메시지 제거]
    DROP --> HIST
    CHECK -- 아니오 --> KEEP[현재 history 유지]
    KEEP --> LOAD["load_memory_variables({})"]
    LOAD --> VIEW[토큰 제한 내 메시지 출력]
```



### 대화 저장 및 확인

여러 대화를 저장하고, 토큰 제한에 따라 어떤 메시지가 유지되는지 확인해봅시다.


In [5]:
# ============================================================
# TODO: 6개의 대화를 저장하고 토큰 제한으로 일부만 남는지 확인하세요
# 힌트: for 루프로 conversations 순회 → save_context() 호출
# 예상 결과: 6개 대화 저장 후 토큰 합계가 150 이내인 최근 메시지만 남음
# ============================================================

# 여러 대화를 순차적으로 저장
conversations = [
    {
        "human": "안녕하세요! 온라인 강의 플랫폼을 구축하고 싶어요. 어떤 기능이 필요할까요?",
        "ai": "안녕하세요! 온라인 강의 플랫폼 구축을 도와드리겠습니다. 기본적으로 필요한 기능은 다음과 같습니다: 1) 동영상 강의 재생, 2) 수강생 관리, 3) 과제 제출 및 평가, 4) 질문과 답변 게시판입니다."
    },
    {
        "human": "동영상 강의 재생 기능은 어떻게 구현하나요?",
        "ai": "동영상 강의 재생 기능은 HTML5 비디오 플레이어를 사용하거나, YouTube API나 Vimeo API를 활용할 수 있습니다. 또한 진행률 추적, 재생 속도 조절, 자막 기능 등을 추가하면 좋습니다."
    },
    {
        "human": "수강생 관리 시스템은 어떤 기능이 필요한가요?",
        "ai": "수강생 관리 시스템에는 회원가입/로그인, 강의 등록, 학습 진도 관리, 수료증 발급 등의 기능이 필요합니다. 또한 관리자 페이지에서 수강생 목록 조회, 통계 분석 기능도 유용합니다."
    },
    {
        "human": "과제 제출 기능은 어떻게 만들면 좋을까요?",
        "ai": "과제 제출 기능은 파일 업로드, 제출 기한 설정, 제출 이력 관리, 교수자 평가 및 피드백 기능이 필요합니다. 또한 제출물 다운로드, 평가 점수 관리 기능도 포함하면 좋습니다."
    },
    {
        "human": "질문과 답변 게시판은 어떤 구조로 만들면 좋을까요?",
        "ai": "질문과 답변 게시판은 질문 작성, 답변 작성, 좋아요/추천 기능, 검색 기능, 카테고리 분류 기능이 필요합니다. 또한 실시간 알림 기능을 추가하면 사용자 경험이 향상됩니다."
    },
    {
        "human": "감사합니다! 도움이 많이 되었어요.",
        "ai": "천만에요! 추가로 궁금한 점이 있으시면 언제든지 문의해주세요. 좋은 하루 되세요!"
    }
]

# 모든 대화 저장 (토큰 합계가 max_token_limit 초과 시 오래된 메시지 자동 삭제)

for conv in conversations:
    memory.save_context(
        inputs={"human": conv["human"]},
        outputs={"ai": conv["ai"]}
    )


In [6]:
# ---------------------------------------------------
# 토큰 제한 후 남은 메시지 확인
# ---------------------------------------------------
# 토큰 초과 시 오래된 메시지부터 삭제되므로 최신 메시지만 남음


stored_messages = memory.load_memory_variables({})["history"]
print(f' ==> [Line 11]: \033[38;2;215;33;244m[stored_messages]\033[0m({type(stored_messages).__name__}) = \033[38;2;49;218;221m{stored_messages}\033[0m')


 ==> [Line 11]: [stored_messages](list) = [HumanMessage(content='질문과 답변 게시판은 어떤 구조로 만들면 좋을까요?', additional_kwargs={}, response_metadata={}), AIMessage(content='질문과 답변 게시판은 질문 작성, 답변 작성, 좋아요/추천 기능, 검색 기능, 카테고리 분류 기능이 필요합니다. 또한 실시간 알림 기능을 추가하면 사용자 경험이 향상됩니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='감사합니다! 도움이 많이 되었어요.', additional_kwargs={}, response_metadata={}), AIMessage(content='천만에요! 추가로 궁금한 점이 있으시면 언제든지 문의해주세요. 좋은 하루 되세요!', additional_kwargs={}, response_metadata={})]


### 💡 토큰 제한 값 비교 루프

```mermaid
flowchart LR
    LIMITS["max_token_limit ∈ {50,100,200}"] --> INIT2[메모리 생성]
    INIT2 --> SAVE2[테스트 대화 save_context]
    SAVE2 --> COUNT2[토큰 길이 계산]
    COUNT2 --> TRIM2{제한 초과?}
    TRIM2 -- 예 --> DROP2[오래된 메시지 제거]
    TRIM2 -- 아니오 --> HOLD2[현재 상태 유지]
    DROP2 --> COUNT2
    HOLD2 --> LOAD2["load_memory_variables({})"]
    LOAD2 --> PRINT2[저장된 메시지 수 출력]
```



💡 **확인 결과**: 총 6개의 대화를 저장했지만, `max_token_limit=150`으로 설정했기 때문에  
토큰 수가 150을 초과하지 않도록 오래된 메시지부터 자동으로 삭제되었습니다.  
최근 메시지들만 저장되어 있어 토큰 제한을 준수하고 있습니다.


## 2. 토큰 제한 값에 따른 차이 비교

다양한 `max_token_limit` 값으로 설정했을 때 어떤 메시지가 저장되는지 비교해봅시다.


In [7]:
# ============================================================
# TODO: max_token_limit=50, 100, 200 세 가지 값으로 메모리를 생성해 비교하세요
# 힌트: for token_limit in [50, 100, 200]: 루프 → 메모리 생성 → 대화 저장 → 결과 출력
# 예상 결과: 제한이 클수록 더 많은 메시지가 유지됨
# ============================================================

# 테스트용 대화 데이터 (길이가 다른 3개 대화)
test_conversations = [
    {"human": "첫 번째 질문입니다. 이것은 비교적 짧은 메시지입니다.", "ai": "첫 번째 답변입니다. 이것도 짧은 답변입니다."},
    {"human": "두 번째 질문입니다. 이번에는 조금 더 긴 메시지를 작성해보겠습니다. 여러 문장으로 구성된 메시지입니다.", "ai": "두 번째 답변입니다. 이 답변도 여러 문장으로 구성되어 있습니다. 더 많은 정보를 포함하고 있습니다."},
    {"human": "세 번째 질문입니다.", "ai": "세 번째 답변입니다."},
]

# 다양한 max_token_limit 값으로 테스트
# - 같은 대화라도 토큰 제한이 다르면 남는 메시지 수가 달라짐

# 다양한 max_token_limit 값으로 테스트
# - 같은 대화라도 토큰 제한이 다르면 남는 메시지 수가 달라짐

for token_limit in [50, 100, 200]:
    print(f"\n[max_token_limit={token_limit}] 테스트 시작")
    
    # 1. 각 제한값으로 메모리 초기화
    memory = ConversationTokenBufferMemory(
        llm=llm, 
        max_token_limit=token_limit, 
        return_messages=True
    )
    
    # 2. 테스트 대화 저장
    for conv in test_conversations:
        memory.save_context(
            inputs={"human": conv["human"]},
            outputs={"ai": conv["ai"]}
        )
    
    # 3. 결과 확인
    history = memory.load_memory_variables({})["history"]
    print(f"✅ 남은 메시지 수: {len(history)}개")
    
    for msg in history:
        role = "사용자" if msg.type == "human" else "AI"
        print(f"   - {role}: {msg.content[:40]}...")
    print("-" * 50)




[max_token_limit=50] 테스트 시작
✅ 남은 메시지 수: 2개
   - 사용자: 세 번째 질문입니다....
   - AI: 세 번째 답변입니다....
--------------------------------------------------

[max_token_limit=100] 테스트 시작
✅ 남은 메시지 수: 4개
   - 사용자: 두 번째 질문입니다. 이번에는 조금 더 긴 메시지를 작성해보겠습니다. 여...
   - AI: 두 번째 답변입니다. 이 답변도 여러 문장으로 구성되어 있습니다. 더 많...
   - 사용자: 세 번째 질문입니다....
   - AI: 세 번째 답변입니다....
--------------------------------------------------

[max_token_limit=200] 테스트 시작
✅ 남은 메시지 수: 6개
   - 사용자: 첫 번째 질문입니다. 이것은 비교적 짧은 메시지입니다....
   - AI: 첫 번째 답변입니다. 이것도 짧은 답변입니다....
   - 사용자: 두 번째 질문입니다. 이번에는 조금 더 긴 메시지를 작성해보겠습니다. 여...
   - AI: 두 번째 답변입니다. 이 답변도 여러 문장으로 구성되어 있습니다. 더 많...
   - 사용자: 세 번째 질문입니다....
   - AI: 세 번째 답변입니다....
--------------------------------------------------


### 💡 윈도우 vs 토큰 메모리 비교 흐름

```mermaid
flowchart TD
    DATA[[comparison_conversations]] --> SAVE_WIN["save_context → window_memory(k=2)"]
    DATA --> SAVE_TOK["save_context → token_memory(limit=100)"]

    SAVE_WIN --> HIST_WIN[대화 개수 기준 유지]
    SAVE_TOK --> HIST_TOK[토큰 길이 기준 유지]

    HIST_WIN --> LOAD_WIN["load_memory_variables({})"]
    HIST_TOK --> LOAD_TOK["load_memory_variables({})"]

    LOAD_WIN --> REPORT_WIN[항상 최근 2개 대화 출력]
    LOAD_TOK --> REPORT_TOK[메시지 길이에 따라 개수 변동]
```



## 3. ConversationBufferWindowMemory와의 차이

`ConversationBufferWindowMemory`는 대화 개수(`k`)로 제한하지만,  
`ConversationTokenBufferMemory`는 토큰 수로 제한합니다.


### 💡 기술 지원 시나리오 토큰 흐름 (limit=200)

```mermaid
flowchart TD
    subgraph SupportChat[기술 지원 대화 반복]
        ISSUE[사용자 이슈 설명] --> SAVE_S["save_context()"]
        SAVE_S --> AGENT[지원팀 응답 저장]
        AGENT --> HIST_S[support_memory.history]
        HIST_S --> TOK_S[토큰 길이 계산]
        TOK_S --> CHECK_S{200 초과?}
        CHECK_S -- 예 --> DROP_S[가장 오래된 메시지 삭제]
        DROP_S --> HIST_S
        CHECK_S -- 아니오 --> KEEP_S[현재 상태 유지]
    end

    KEEP_S --> LOAD_S["load_memory_variables({})"]
    LOAD_S --> VIEW_S[토큰 제한 내 대화 출력]
```



In [ ]:
# ---------------------------------------------------
# ConversationBufferWindowMemory vs ConversationTokenBufferMemory 비교
# ---------------------------------------------------
# 길이가 서로 다른 메시지를 섞어 저장하면 두 방식의 차이가 명확히 드러남
# - WindowMemory: 항상 같은 개수(k=2)를 유지
# - TokenBufferMemory: 메시지 길이에 따라 보관 개수가 달라짐

from langchain.memory import ConversationBufferWindowMemory

# 비교용 대화 데이터 (길이가 다른 메시지들)
comparison_conversations = [
    {"human": "짧은 질문", "ai": "짧은 답변"},
    {"human": "이것은 중간 길이의 질문입니다. 여러 문장으로 구성되어 있습니다.", "ai": "이것은 중간 길이의 답변입니다. 여러 문장으로 구성되어 있습니다."},
    {"human": "매우 긴 질문입니다. " * 5, "ai": "매우 긴 답변입니다. " * 5},
    {"human": "또 다른 질문", "ai": "또 다른 답변"},
]

# ConversationBufferWindowMemory: k=2 (최근 2개 대화)
# - 메시지 길이와 무관하게 항상 최근 2턴 유지


# ConversationTokenBufferMemory: max_token_limit=100 (약 2개 대화 분량)
# - 메시지가 길면 제한에 더 빨리 도달하여 더 적게 유지됨


# 두 메모리에 동일한 대화 저장


# 결과 비교


## 4. 실전 예제: 기술 지원 챗봇

토큰 제한이 중요한 기술 지원 시나리오에서 `ConversationTokenBufferMemory`를 사용하는 예제입니다.


## 5. max_token_limit 값 선택 가이드

적절한 `max_token_limit` 값을 선택하는 방법을 알아봅시다.


In [ ]:
print("=" * 60)
print("📋 max_token_limit 값 선택 가이드")
print("=" * 60)

guidelines = [
    {
        "limit": 50,
        "설명": "매우 짧은 대화만 유지",
        "장점": "토큰 사용량 최소화",
        "단점": "컨텍스트가 매우 제한적",
        "적용": "간단한 질의응답, 토큰 예산이 매우 제한적인 경우"
    },
    {
        "limit": 100,
        "설명": "짧은 대화 유지",
        "장점": "토큰 효율적이면서 최소한의 컨텍스트 유지",
        "단점": "여전히 제한적인 컨텍스트",
        "적용": "짧은 대화, 간단한 상담"
    },
    {
        "limit": 200,
        "설명": "중간 길이 대화 유지",
        "장점": "적절한 컨텍스트와 토큰 효율성의 균형",
        "단점": "복잡한 대화에서는 부족할 수 있음",
        "적용": "일반적인 고객 상담, 중간 길이 대화"
    },
    {
        "limit": 500,
        "설명": "긴 대화 유지",
        "장점": "충분한 컨텍스트 제공",
        "단점": "토큰 사용량 증가",
        "적용": "복잡한 상담, 긴 대화"
    }
]

for guide in guidelines:
    print(f"\n🔹 max_token_limit={guide['limit']}: {guide['설명']}")
    print(f"   ✅ 장점: {guide['장점']}")
    print(f"   ⚠️  단점: {guide['단점']}")
    print(f"   💡 적용: {guide['적용']}")

print("\n" + "=" * 60)
print("💡 권장사항")
print("=" * 60)
print("• 짧은 대화: 50~100 토큰")
print("• 일반적인 대화: 100~200 토큰")
print("• 복잡한 대화: 200~500 토큰")
print("• 매우 긴 대화: ConversationSummaryMemory 고려")
print("\n⚠️  주의: 사용하는 LLM의 최대 토큰 제한을 고려하여 설정하세요!")


## 핵심 정리

```python
# ConversationTokenBufferMemory 기본 사용법
from langchain.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=150,
    return_messages=True
)

# 대화 저장
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 대화 로드 (토큰 제한 내에서만 반환)
history = memory.load_memory_variables({})["history"]
```

### 주요 특징
- ✅ **토큰 기반 제한**: `max_token_limit`으로 최대 토큰 수 지정
- ✅ **정확한 토큰 관리**: LLM의 토큰 제한을 정확하게 준수
- ✅ **자동 플러시**: 토큰 제한 초과 시 오래된 메시지부터 자동 삭제
- ✅ **동적 관리**: 메시지 길이에 따라 유연하게 관리

### ConversationBufferWindowMemory와의 차이

| 특징 | ConversationBufferWindowMemory | ConversationTokenBufferMemory |
|------|-------------------------------|-------------------------------|
| 제한 기준 | 대화 개수 (k) | 토큰 수 (max_token_limit) |
| 정확도 | 대략적 (메시지 길이 무관) | 정확 (토큰 수 기준) |
| 유연성 | 메시지 길이와 무관하게 일정 | 메시지 길이에 따라 변동 |
| 적합한 경우 | 대화 개수로 관리 가능한 경우 | 정확한 토큰 관리가 필요한 경우 |

### 주의사항
- ⚠️ **LLM 필요**: 토큰 계산을 위해 LLM 인스턴스가 필요함
- ⚠️ **토큰 계산 비용**: 토큰 계산에 약간의 오버헤드 발생
- ⚠️ **max_token_limit 선택**: 너무 작으면 컨텍스트 부족, 너무 크면 토큰 낭비